# Fisher matrix with a custom noise model

This example shows how to give StableEMRIFisher your own noise model (PSD). As a simple test, it defines a noise model with an extra *degradation factor* $d$ that scales the LISA PSD, $S_n(f) \to d\, S_n(f)$, and checks that the SNR changes as expected.

Any callable can be used as a noise model, as long as:

- its first argument is the frequency, and
- any other settings are keyword arguments, passed through `noise_kwargs` (one dictionary per channel).

**Requirements:** `lisaanalysistools` and `fastlisaresponse`, for the LISA response and noise model (see the installation guide).

*Validated with LISAAnalysisTools-cuda12x v1.2.0 and fastlisaresponse v1.1.17.*

In [1]:
import numpy as np

import few
from few.waveform import FastSchwarzschildEccentricFlux
from fastlisaresponse import ResponseWrapper
from lisatools.detector import EqualArmlengthOrbits
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens

from stableemrifisher.fisher import StableEMRIFisher

use_gpu = False  # set to True to run on an NVIDIA GPU

if not use_gpu:
    # restrict FEW to its CPU backend
    cfg_set = few.get_config_setter(reset=True)
    cfg_set.enable_backends("cpu")

## Define the custom noise model

The noise model wraps `lisatools.sensitivity.get_sensitivity` and multiplies it by the degradation factor `d`.

In [2]:
def degraded_sensitivity(f, d=1.0, **noise_kwargs):
    """LISA PSD at frequency f (Hz), multiplied by the degradation factor d."""
    return d * get_sensitivity(f, **noise_kwargs)

## Set up two Fisher calculators

The setup is the same as in `fisher_matrix_with_response.ipynb`, except for the noise model:

- **Waveform:** FEW's Schwarzschild eccentric flux model.
- **Response:** first-generation TDI on equal-arm-length orbits, returning the A and E channels. `index_lambda` and `index_beta` tell the response which waveform arguments hold the sky position (`phiS` and `qS`); the first and last `t0` seconds of the signal are set to zero.
- **Noise:** the LISA PSD of each TDI channel, from `get_sensitivity` with the matching `sens_fn` (one dictionary of `noise_kwargs` per channel).

Both calculators use `degraded_sensitivity`; one leaves the PSD as it is ($d = 1$, the default) and the other degrades it by a factor of 2 in each channel.

In [3]:
T = 0.1  # observation time, years
dt = 10.0  # sampling interval, seconds

ResponseWrapper_kwargs = dict(
    Tobs=T,
    dt=dt,
    index_lambda=8,  # position of phiS in the waveform arguments
    index_beta=7,  # position of qS in the waveform arguments
    t0=10000.0,  # seconds at each end of the signal where the response is unreliable
    remove_garbage="zero",  # zero the signal within t0 of each end
    flip_hx=True,  # FEW returns h+ - i hx
    is_ecliptic_latitude=False,  # qS is a polar angle, not a latitude
    orbits=EqualArmlengthOrbits(),
    order=20,  # order of the Lagrange interpolation of the time delays
    tdi="1st generation",
    tdi_chan="AE",
)

channels = [A1TDISens, E1TDISens]


def make_sef(noise_kwargs):
    """StableEMRIFisher with the custom noise model and the given noise kwargs."""
    return StableEMRIFisher(
        waveform_class=FastSchwarzschildEccentricFlux,
        waveform_class_kwargs=dict(
            inspiral_kwargs=dict(err=1e-11),  # trajectory integrator tolerance
            mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
        ),
        ResponseWrapper=ResponseWrapper,
        ResponseWrapper_kwargs=ResponseWrapper_kwargs,
        noise_model=degraded_sensitivity,
        noise_kwargs=noise_kwargs,
        channels=channels,
        use_gpu=use_gpu,
    )


sef_normal = make_sef([{"sens_fn": channel} for channel in channels])
sef_degraded = make_sef([{"sens_fn": channel, "d": 2.0} for channel in channels])

## Compare the SNRs

Compute the SNR of the same source with each noise model.

In [4]:
wave_params = dict(
    m1=1e6,  # primary mass, solar masses
    m2=10.0,  # secondary mass, solar masses
    a=0.0,  # primary spin
    p0=9.5,  # initial semi-latus rectum
    e0=0.4,  # initial eccentricity
    xI0=1.0,  # cosine of the initial inclination
    dist=0.1,  # luminosity distance, Gpc
    qS=np.pi / 3,  # sky location, polar angle
    phiS=np.pi / 4,  # sky location, azimuthal angle
    qK=np.pi / 6,  # spin orientation, polar angle
    phiK=np.pi / 8,  # spin orientation, azimuthal angle
    Phi_phi0=1.0,  # initial azimuthal phase
    Phi_theta0=0.0,  # initial polar phase
    Phi_r0=0.0,  # initial radial phase
)

snr_normal = sef_normal.compute_snr(*wave_params.values(), dt=dt, T=T, use_gpu=use_gpu)
snr_degraded = sef_degraded.compute_snr(*wave_params.values(), dt=dt, T=T, use_gpu=use_gpu)

print(f"SNR with the normal PSD:   {snr_normal:.2f}")
print(f"SNR with the degraded PSD: {snr_degraded:.2f}")
print(f"ratio: {snr_normal / snr_degraded:.4f}  (expected sqrt(2) = {np.sqrt(2):.4f})")

SNR with the normal PSD:   237.84
SNR with the degraded PSD: 168.18
ratio: 1.4142  (expected sqrt(2) = 1.4142)


## What this shows

The squared SNR is $\rho^2 = (h|h) \propto \int |\tilde h(f)|^2 / S_n(f)\, df$, so scaling the PSD by $d$ scales the SNR by $1/\sqrt{d}$. With $d = 2$ the ratio of the two SNRs is $\sqrt{2}$, as found above.

The same `noise_model` and `noise_kwargs` arguments are used for the Fisher matrix: calling `sef_degraded(wave_params, param_names=[...])` computes it with the degraded PSD.